# COMP64702 RAG Coursework
## Notebook 1: Corpus and Benchmark
Chosen cuisine: South Asian cuisine

This notebook loads the background corpus and benchmark dataset,
then expands the corpus by scraping missing Wikipedia documents
and enriching stub documents with full content.

In [19]:
# Imports

import json, requests, numpy, pandas as pd, torch
print("Python:", __import__('sys').version)
print("Torch:", torch.__version__)
print("All imports OK ✅")

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]
Torch: 2.11.0
All imports OK ✅


In [20]:
# File paths

corpus_path = "background_corpus.json"
benchmark_path = "benchmark_dataset.json"
benchmark_input_path = "benchmark_input_only.json"

In [21]:
# Load corpus

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} corpus documents")

Loaded 17 corpus documents


In [22]:
# Show corpus summary

corpus_df = pd.DataFrame([
    {
        "doc_id": d.get("doc_id"),
        "title": d.get("title"),
        "source": d.get("source"),
        "char_count": d.get("char_count", len(d.get("full_text", "")))
    }
    for d in corpus
])

corpus_df

,doc_id,title,source,char_count
0,wiki_sri_lankan_cuisine,Sri Lankan cuisine,wikipedia,10402
1,wiki_naan,Naan,wikipedia,4690
2,wiki_turmeric,Turmeric,wikipedia,14629
3,wiki_cumin,Cumin,wikipedia,12384
4,wiki_cardamom,Cardamom,wikipedia,13295
5,wiki_indian_cuisine,Indian cuisine,wikipedia,2278
6,wiki_pakistani_cuisine,Pakistani cuisine,wikipedia,1603
7,wiki_bangladeshi_cuisine,Bangladeshi cuisine,wikipedia,1311
8,wiki_nepali_cuisine,Nepali cuisine,wikipedia,1248
9,wiki_biryani,Biryani,wikipedia,1301


In [23]:
# Show one example document

example_doc = corpus[0]
print("TITLE:", example_doc["title"])
print("DOC ID:", example_doc["doc_id"])
print("SOURCE:", example_doc["source"])
print()
print(example_doc["full_text"][:1500])

TITLE: Sri Lankan cuisine
DOC ID: wiki_sri_lankan_cuisine
SOURCE: wikipedia

Sri Lankan cuisine is known for its particular combinations of herbs, spices, fish, vegetables, rices , and fruits. The cuisine is highly centered around many varieties of rice, as well as coconut which is a ubiquitous plant throughout the country. Seafood also plays a significant role in the cuisine, be it fresh fish or preserved fish. As a country that was a hub in the historic oceanic silk road , contact with foreign traders brought new food items and cultural influences in addition to the local traditions of the country's ethnic groups, all of which have helped shape Sri Lankan cuisine. Influences from Indian (particularly South Indian ), Indonesian and Dutch cuisines are most evident with Sri Lankan cuisine sharing close ties to other neighbouring South and Southeast Asian cuisines .

Sri Lanka is historically famous for its cinnamon. The 'true cinnamon' tree, or Cinnamomum verum , used to be botanically 

In [24]:
# Load benchmark dataset

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

print(f"Loaded {len(benchmark)} benchmark questions")

Loaded 13 benchmark questions


In [25]:
# Show benchmark summary

benchmark_df = pd.DataFrame(benchmark)
benchmark_df[["query_id", "query", "answer_source", "question_type", "difficulty"]]

,query_id,query,answer_source,question_type,difficulty
0,sa_000,What is biryani and which region is it origina...,wiki_biryani,factual,easy
1,sa_001,What distinguishes Hyderabadi biryani from Luc...,wiki_biryani,comparative,hard
2,sa_002,What is the difference between naan and roti?,wiki_naan,comparative,easy
3,sa_003,How is dosa batter traditionally prepared?,wiki_dosa,procedural,medium
4,sa_004,What ingredients are used in a classic samosa ...,wiki_samosa,procedural,medium
5,sa_005,What is cardamom and what are the differences ...,wiki_cardamom,ingredient,hard
6,sa_006,How does Sri Lankan cuisine differ from South ...,wiki_sri_lankan_cuisine,comparative,hard
7,sa_007,What is the difference between a curry and a m...,wiki_curry,comparative,medium
8,sa_008,What are the key features of Pakistani cuisine...,wiki_pakistani_cuisine,comparative,hard
9,sa_009,What is butter chicken and where did it origin...,wiki_butter_chicken,factual,easy


In [26]:
# Load benchmark input-only file

with open(benchmark_input_path, "r", encoding="utf-8") as f:
    benchmark_input = json.load(f)

print(f"Loaded {len(benchmark_input['queries'])} input-only queries")
benchmark_input["queries"][:3]

Loaded 13 input-only queries


[{'query_id': 'sa_000',
  'query': 'What is biryani and which region is it originally associated with?'},
 {'query_id': 'sa_001',
  'query': 'What distinguishes Hyderabadi biryani from Lucknowi (Awadhi) biryani?'},
 {'query_id': 'sa_002',
  'query': 'What is the difference between naan and roti?'}]

### ── Corpus Expansion ───────
The original corpus had 17 documents, but several benchmark queries had no matching source document (e.g. butter_chicken), causing the LLM to hallucinate answers. I added the cell below which defines a Wikipedia scraper to fix that.

In [ ]:



import time, requests

def scrape_wikipedia(wiki_title, doc_id):
    headers = {"User-Agent": "RAG-Coursework/1.0"}
    parse_url = (
        f"https://en.wikipedia.org/w/api.php"
        f"?action=query&titles={wiki_title.replace(' ', '_')}"
        f"&prop=extracts&explaintext=true&format=json"
    )
    resp = requests.get(parse_url, headers=headers, timeout=15)
    pages = resp.json().get("query", {}).get("pages", {})
    full_text, title = "", wiki_title
    if pages:
        page = next(iter(pages.values()))
        full_text = page.get("extract", "")[:15000]
        title = page.get("title", wiki_title)

    sections, current_heading, current_content = [], "Introduction", []
    for line in full_text.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("==") and line.endswith("=="):
            if current_content:
                sections.append({"heading": current_heading, "content": current_content})
            current_heading, current_content = line.strip("= ").strip(), []
        elif len(line) > 30:
            current_content.append(line)
    if current_content:
        sections.append({"heading": current_heading, "content": current_content})
    if not sections and full_text:
        sections = [{"heading": "Introduction", "content": [full_text[:3000]]}]

    return {
        "doc_id": doc_id, "title": title,
        "url": f"https://en.wikipedia.org/wiki/{wiki_title.replace(' ', '_')}",
        "source": "wikipedia", "cuisine": "south_asian",
        "full_text": full_text, "sections": sections,
        "char_count": len(full_text),
        "scraped_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }

print("Scraper ready ✅")

Scraper ready ✅


In [28]:
# Reload original corpus as the base for expansion
with open("background_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)
print(f"Loaded {len(corpus)} documents ✅")

Loaded 17 documents ✅


In [29]:
# Documents completely missing from your corpus
NEW_DOCS = [
    ("Butter_chicken",     "wiki_butter_chicken"),
    ("Hyderabadi_biryani", "wiki_hyderabadi_biryani"),
    ("Lucknawi_cuisine",   "wiki_lucknawi_cuisine"),
    ("Masala",             "wiki_masala"),
    ("Garam_masala",       "wiki_garam_masala"),
    ("Mughlai_cuisine",    "wiki_mughlai_cuisine"),
]

# Stub documents that need full content
ENRICH_DOCS = [
    ("Biryani",          "wiki_biryani"),
    ("Dal",              "wiki_dal"),
    ("Roti",             "wiki_roti"),
    ("Dosa_(food)",      "wiki_dosa"),
    ("Samosa",           "wiki_samosa"),
    ("Tandoori_chicken", "wiki_tandoori_chicken"),
    ("Curry",            "wiki_curry"),
    ("Pilaf",            "wiki_pilaf"),
]

existing_ids = {doc["doc_id"] for doc in corpus}

# Scrape new docs
new_docs = []
for wiki_title, doc_id in NEW_DOCS:
    if doc_id in existing_ids:
        print(f"SKIP {doc_id}")
        continue
    print(f"Scraping {wiki_title}...", end=" ")
    try:
        doc = scrape_wikipedia(wiki_title, doc_id)
        new_docs.append(doc)
        print(f"OK ({doc['char_count']} chars)")
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(0.5)

# Enrich stub docs
enriched_docs = {}
for wiki_title, doc_id in ENRICH_DOCS:
    print(f"Enriching {doc_id}...", end=" ")
    try:
        doc = scrape_wikipedia(wiki_title, doc_id)
        enriched_docs[doc_id] = doc
        print(f"OK ({doc['char_count']} chars)")
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(0.5)

print(f"\nDone! {len(new_docs)} new docs, {len(enriched_docs)} enriched")

Scraping Butter_chicken... OK (2662 chars)
Scraping Hyderabadi_biryani... OK (3180 chars)
Scraping Lucknawi_cuisine... OK (0 chars)
Scraping Masala... OK (1369 chars)
Scraping Garam_masala... OK (2291 chars)
Scraping Mughlai_cuisine... OK (9045 chars)
Enriching wiki_biryani... OK (11889 chars)
Enriching wiki_dal... OK (7539 chars)
Enriching wiki_roti... OK (11557 chars)
Enriching wiki_dosa... OK (6781 chars)
Enriching wiki_samosa... OK (8013 chars)
Enriching wiki_tandoori_chicken... OK (3558 chars)
Enriching wiki_curry... OK (15000 chars)
Enriching wiki_pilaf... OK (15000 chars)

Done! 6 new docs, 8 enriched


In [30]:
# Build expanded corpus
expanded_corpus = []
for doc in corpus:
    expanded_corpus.append(enriched_docs.get(doc["doc_id"], doc))
expanded_corpus.extend(new_docs)

print(f"Corpus: {len(corpus)} → {len(expanded_corpus)} documents")

# Save it
with open("background_corpus_expanded.json", "w", encoding="utf-8") as f:
    json.dump(expanded_corpus, f, indent=2, ensure_ascii=False)

print("Saved background_corpus_expanded.json ✅")

Corpus: 17 → 23 documents
Saved background_corpus_expanded.json ✅


In [31]:
with open("benchmark_dataset.json", "r", encoding="utf-8") as f:
    benchmark = json.load(f)

expanded_ids = {doc["doc_id"] for doc in expanded_corpus}

print("Coverage check:")
all_ok = True
for item in benchmark:
    src = item["answer_source"]
    status = "✅" if src in expanded_ids else "❌ MISSING"
    if src not in expanded_ids:
        all_ok = False
    print(f"  {item['query_id']} | {src:<30} {status}")

print("\n✅ All covered!" if all_ok else "\n❌ Still missing some docs — tell me which ones")

Coverage check:
  sa_000 | wiki_biryani                   ✅
  sa_001 | wiki_biryani                   ✅
  sa_002 | wiki_naan                      ✅
  sa_003 | wiki_dosa                      ✅
  sa_004 | wiki_samosa                    ✅
  sa_005 | wiki_cardamom                  ✅
  sa_006 | wiki_sri_lankan_cuisine        ✅
  sa_007 | wiki_curry                     ✅
  sa_008 | wiki_pakistani_cuisine         ✅
  sa_009 | wiki_butter_chicken            ✅
  sa_010 | wiki_tandoori_chicken          ✅
  sa_011 | wiki_dal                       ✅
  sa_012 | wiki_biryani                   ✅

✅ All covered!


In [32]:
print(f"""
State:
- Original corpus: {len(corpus)} documents
- Expanded corpus: {len(expanded_corpus)} documents  
- New docs added: {len(new_docs)}
- Enriched stubs: {len(enriched_docs)}
- Benchmark questions: {len(benchmark)}
- Coverage: all {len(benchmark)} questions have a source document ✅
""")


State:
- Original corpus: 17 documents
- Expanded corpus: 23 documents  
- New docs added: 6
- Enriched stubs: 8
- Benchmark questions: 13
- Coverage: all 13 questions have a source document ✅

